In [ ]:
# Instalar o actualizar las librerías necesarias
!pip install -U transformers accelerate

In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df_train = pd.read_csv('/content/drive/MyDrive/PLN/subtask1/train.csv')
df_dev = pd.read_csv('/content/drive/MyDrive/PLN/subtask1/devel.csv')

Mounted at /content/drive


Entrenar MarIA

In [ ]:

import numpy as np
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn # Necesario si vas a modificar directamente la función de pérdida del modelo

# Definimos los nombres de tus categorías para que el reporte sea legible
# Sustituye los nombres en orden del 0 al 3
nombres_clases = ["Mild", "Medium", "High", "Severe"]
id2label = {i: nombre for i, nombre in enumerate(nombres_clases)}
label2id = {nombre: i for i, nombre in enumerate(nombres_clases)}

def preparar_dataset(df):
    # Renombramos la columna para que el Trainer la reconozca automáticamente
    # Asegúrate de que el nombre de la columna de clase sea 'CLASS' en tu DataFrame
    return Dataset.from_pandas(df[['TEXT', 'CLASS']].rename(columns={'CLASS': 'label'}))

dataset = DatasetDict({
    'train': preparar_dataset(df_train),
    'validation': preparar_dataset(df_dev)
})

# 2. CONFIGURACIÓN DEL MODELO BASE
model_checkpoint = "BSC-LT/MrBERT-es"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_fn(examples):
    return tokenizer(examples["TEXT"], truncation=True, padding=False, max_length=512)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Dataset tokenizado y data collator preparado.")

Map:   0%|          | 0/13630 [00:00<?, ? examples/s]

Map:   0%|          | 0/3405 [00:00<?, ? examples/s]

Dataset tokenizado y data collator preparado.


In [ ]:
# 3. MÉTRICAS
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

print("Métricas F1 y Accuracy cargadas y función compute_metrics definida.")

Métricas F1 y Accuracy cargadas y función compute_metrics definida.


In [ ]:
# 3.1. CALCULAR PESOS DE CLASE PARA CLASES DESBALANCEADAS
# Extraer las etiquetas numéricas del conjunto de entrenamiento
train_labels = tokenized_dataset["train"]["label"]

# Definir todas las clases posibles en orden (0, 1, 2, 3)
unique_labels_in_order = np.array(sorted(id2label.keys()))

# Calcular los pesos de clase
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=unique_labels_in_order,
    y=train_labels
)

# Convertir los pesos de clase a un tensor de PyTorch
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32)

print("Pesos de clase calculados:", class_weights_tensor)

Pesos de clase calculados: tensor([1.1527, 0.7192, 0.6405, 5.5317])


Ahora que tenemos los pesos de clase, podemos crear una subclase de `Trainer` que los incorpore en la función de pérdida. Esto le dará más importancia a las clases minoritarias durante el entrenamiento.

In [ ]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0): # Added num_items_in_batch
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 4. MODELO
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    force_download=True # Agregado para forzar la descarga de los archivos del modelo
)

# 5. ARGUMENTOS DE ENTRENAMIENTO
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/PLN/maria_clasificador",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.25,
    warmup_ratio=0.2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    push_to_hub=False,
    label_smoothing_factor=0.15,
    fp16=True,
    report_to="none" # Evita pedir logs externos si no los tienes configurados
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 6. LANZAR ENTRENAMIENTO
trainer.train()

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/601M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4260, training_loss=0.5411334365746225, metrics={'train_runtime': 6549.2675, 'train_samples_per_second': 20.811, 'train_steps_per_second': 0.65, 'total_flos': 4.471988945089224e+16, 'train_loss': 0.5411334365746225, 'epoch': 10.0})

In [ ]:
model_path = '/content/drive/MyDrive/PLN/maria_clasificador/checkpoint-4260'
print(f"model_path configurado a: {model_path}")

model_path configurado a: /content/drive/MyDrive/PLN/maria_clasificador/checkpoint-4260


In [ ]:
print(f"Cargando tokenizer y modelo desde: {model_path}")

tokenizer_loaded = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model_loaded = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

print("Tokenizador y modelo cargados exitosamente.")

Cargando tokenizer y modelo desde: /content/drive/MyDrive/PLN/maria_clasificador/checkpoint-4260


Loading weights:   0%|          | 0/138 [00:01<?, ?it/s]

Tokenizador y modelo cargados exitosamente.


In [ ]:
df_test = pd.read_csv('/content/drive/MyDrive/PLN/subtask1/test.csv', header=None, names=['TEXT'])
print("Dataset de prueba cargado.")
display(df_test.head())

Dataset de prueba cargado.


,TEXT
0,REFIERE USUARIA QUE EL DIA DE AYER JUEVES DE F...
1,LA USUARIA MANIFIESTA QUE EL SEÑOR LLEGA AL DO...
2,US. REFIERE QUE HACE UN MES APROX. LE MARCO PO...
3,USURIA ACUDE A TRAMITE DE PENSION ALIMENTICIA
4,"LLEGO EL AGRESOR SE METE AL PATIO DE ATRAS, LE..."


Ahora que el modelo y el tokenizador están cargados, podemos probar a hacer predicciones en el dataset de prueba.

In [ ]:
import torch

# Redefine id2label to ensure it's available
nombres_clases = ["Mild", "Medium", "High", "Severe"]
id2label = {i: nombre for i, nombre in enumerate(nombres_clases)}

def predict_class(text):
    inputs = tokenizer_loaded(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model_loaded.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_loaded(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    return id2label[predictions.item()]

# Aplicar la función de predicción al dataset de prueba
df_test['PREDICTED_CLASS'] = df_test['TEXT'].apply(predict_class)

print("Predicciones realizadas en el dataset de prueba.")
display(df_test.head())

Predicciones realizadas en el dataset de prueba.


,TEXT,PREDICTED_CLASS
0,REFIERE USUARIA QUE EL DIA DE AYER JUEVES DE F...,High
1,LA USUARIA MANIFIESTA QUE EL SEÑOR LLEGA AL DO...,High
2,US. REFIERE QUE HACE UN MES APROX. LE MARCO PO...,Medium
3,USURIA ACUDE A TRAMITE DE PENSION ALIMENTICIA,Mild
4,"LLEGO EL AGRESOR SE METE AL PATIO DE ATRAS, LE...",High


In [ ]:
import os
import zipfile

# Redefine label2id (or ensure it's available from previous cells)
label2id = {nombre: i for i, nombre in enumerate(nombres_clases)}

# Convert predicted class names back to numerical IDs
df_test['PREDICTED_CLASS_ID'] = df_test['PREDICTED_CLASS'].map(label2id)

# Define the output directory in Google Drive
output_dir = '/content/drive/MyDrive/PLN/subtask1_predictions'
os.makedirs(output_dir, exist_ok=True)

# Define the CSV file path
csv_filename = os.path.join(output_dir, 'subtask1.csv')

# Save the predictions to a CSV file without header
df_test[['PREDICTED_CLASS_ID']].to_csv(csv_filename, index=False, header=False)

print(f"Predicciones guardadas en: {csv_filename}")

# Create a zip file
zip_filename = os.path.join(output_dir, 'test_predictions.zip')

with zipfile.ZipFile(zip_filename, 'w') as zf:
    zf.write(csv_filename, os.path.basename(csv_filename))

print(f"Archivo ZIP creado en: {zip_filename}")

Predicciones guardadas en: /content/drive/MyDrive/PLN/subtask1_predictions/subtask1.csv
Archivo ZIP creado en: /content/drive/MyDrive/PLN/subtask1_predictions/test_predictions.zip
